In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import paderbox as pb
import torch

from padertorch.base import Model
from pathlib import Path
from sklearn import datasets

from pvq_manipulation.models.flow_matching import FlowMatching
from pvq_manipulation.models.ode_functions import MLP
from pvq_manipulation.helper.probability_paths import Optimal_Transport, DiffusionVarianceExploding, DiffusionVariancePreserving

# Simple 1D Example
Generate a "multi hill" distribution (e.g., gaussian mixture model)

In [ ]:
mean = torch.tensor([
    [-4.], [0.], [2],  
])
cov = torch.tensor([
    [[0.4]], [[0.25]], [[0.25]],  
])
pi = torch.tensor([0.6, 0.15, 0.25])  

observation_dist = torch.distributions.MixtureSameFamily(
    torch.distributions.Categorical(pi),
    torch.distributions.MultivariateNormal(mean, cov)
)

x = torch.linspace(-6, 4, 100)
plt.figure(figsize=(8, 1))
plt.plot(x.numpy(), torch.exp(observation_dist.log_prob(x[:, None])).numpy());

# train a flow matching model

In [ ]:
latent_dist = torch.distributions.MultivariateNormal(torch.zeros(1), torch.eye(1))

fm = FlowMatching(
    MLP(input_dim=1, condition_dim=0, hidden_channels=[64,64]),
    DiffusionVariancePreserving(beta_min=0.1, beta_max=20.0), #DiffusionVariancePreserving() #DiffusionVarianceExploding()
)
optimizer = torch.optim.Adam(fm.parameters(), lr=1e-3)
batch_size = 128 * 12 

for epoch in range(3000): 
    x = observation_dist.sample((batch_size,))
    x.requires_grad_(True) 
    estimated_vector_field, target_vector_field = fm((x, None))
    
    losses = fm.review(None, (estimated_vector_field, target_vector_field))

    loss = 0
    for key in losses['losses']:
        loss += losses['losses'][key]
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate Data

In [ ]:
z = latent_dist.sample((100_000,))

with torch.no_grad():
    y = fm.sample(z, condition=None, start=0, stop=1, steps=20)
    y = torch.squeeze(y, axis=-1)
bins = torch.linspace(y.amin(), y.amax(), 150)
hist = torch.stack([torch.histogram(y_t, bins=bins, density=True)[0] for y_t in y])

fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(4,1)
axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[1:3, 0]),
    fig.add_subplot(gs[3, 0]),
]

pos = [(l+h)/2 for l, h in zip(bins, bins[1:])]
axes[0].plot(bins, torch.exp(observation_dist.log_prob(bins[:, None])))
axes[0].plot(pos, hist[-1])
axes[0].set_xlim(bins[0], bins[-1])
axes[1].imshow(
    hist,
    origin='lower',
    aspect='auto',
    extent=[bins[0].item(), bins[-1].item(), 0, 1]
)
axes[2].plot(bins, torch.exp(latent_dist.log_prob(bins[:, None])))
axes[2].plot(pos, hist[0])
axes[2].set_xlim(bins[0], bins[-1])
axes[0].set_ylabel(r"$p_x(x)$")
axes[1].set_ylabel(r"$t$")
axes[2].set_ylabel(r"$p_z(z)$")

# example of loading a trained model
First, run the train.py file with the config: config_toy_example_flow_matching

In [ ]:
storage_dir = # Path(path/to/model)

model_dict = pb.io.load_yaml(storage_dir / "config.yaml")

model = Model.from_config(model_dict['trainer']['model'])
cp = torch.load(
    storage_dir / "checkpoints/ckpt_best_loss.pth",
    map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    weights_only=False
)

# remove speaker encoder
model_weights = cp.copy()
model.load_state_dict(model_weights['model'])
model.eval()

# Data distribution

In [ ]:
def plot_distribution(data, labels, title):
    fig = plt.figure(figsize=(5,5))
    plt.title(title)
    if labels is None:
        plt.plot(data[:,0], data[:,1], '.')
    else:
        plt.plot(data[:,0][labels==0], data[:,1][labels==0], '.')
        plt.plot(data[:,0][labels==1], data[:,1][labels==1], '.')
    plt.axis('equal')
    plt.show()

# create dataset
x, y = datasets.make_moons(n_samples=1000, noise=.05)

plot_distribution(
    x, 
    y, 
    title='Data distribution $p_X(x)$',
)

# Transformation from data distribution to base distribution
$z=f(x, c)$ with condition $c$

In [ ]:
with torch.no_grad():
    z = model.sample(
        torch.tensor(x).float(), 
        torch.tensor(y)[:,None],
        start=1,
        stop=0,
    )
plot_distribution(
    z[-1, :], 
    y, 
    title='Data distribution $p_X(x)$',
)

# Transformation from base distribution to target distribution
$x = f^{-1}(z, c)$ with condition $c$

In [ ]:
# base distribution is a unit Gaussian
base_mean = torch.tensor([0,0], dtype=torch.float)
base_std = torch.tensor([[1.0,0],[0,1]], dtype=torch.float)
base = np.random.multivariate_normal(base_mean, base_std, 1000)

class_1_labels = torch.zeros(base.shape[0])
class_2_labels = torch.ones(base.shape[0])

with torch.no_grad():
    generated_class_1 = model.sample(
        torch.tensor(base, dtype=torch.float), 
        class_1_labels[:, None],
        start=0,
        stop=1,
    )

    generated_class_2 = model.sample(
        torch.tensor(base, dtype=torch.float), 
        class_2_labels[:, None],
        start=0,
        stop=1,
    )
    
plot_distribution(
    torch.concatenate((generated_class_1[-1,:], generated_class_2[-1,:])),
    torch.concatenate((class_1_labels, class_2_labels),dim=0),
    'Generated data distribution $P_X(x)$'
)
